# Optimizers & Loss Functions: Training Logic

Reach for this when you need: 
- Selecting the right loss function for your task.
- Configuring AdamW/SGD and standard hyperparameters.
- Implementing custom loss functions.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 1. Common Loss Functions

| Task | Loss Function | Activation Req |
| :--- | :--- | :--- |
| Binary Classification | `nn.BCEWithLogitsLoss` | Raw Logits (preferred over Sigmoid + BCELoss) |
| Multi-class Classification | `nn.CrossEntropyLoss` | Raw Logits (Softmax is applied internally) |
| Regression | `nn.MSELoss` / `nn.L1Loss` | Linear output |
| Object Detection/Robust Reg | `nn.SmoothL1Loss` | Linear output |

In [ ]:
# CrossEntropy: Expects (Batch, Classes) and (Batch) labels
criterion = nn.CrossEntropyLoss()
logits = torch.randn(3, 5, requires_grad=True) # 3 samples, 5 classes
target = torch.empty(3, dtype=torch.long).random_(5)
loss = criterion(logits, target)

# Binary Cross Entropy: Expects (Batch) and (Batch) floats
bce_criterion = nn.BCEWithLogitsLoss()
logits_b = torch.randn(3, requires_grad=True)
target_b = torch.tensor([1., 0., 1.])
loss_b = bce_criterion(logits_b, target_b)

## 2. Optimizers

**AdamW**
Standard adaptive optimizer with decoupled weight decay. Industry default for Transformers.

✅ **Use when**: High-dimensional parameter spaces (LLMs, large models).
❌ **Don't use when**: Fine-tuning carefully on small datasets (SGD might generalize better).

In [ ]:
model = nn.Linear(10, 1)

# lr: Magnitude of updates. weight_decay: L2 regularization.
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)

# Common training step pattern:
optimizer.zero_grad() # Clear old gradients
loss.backward()      # Compute new gradients
optimizer.step()      # Update parameters

## 3. Learning Rate Schedulers

| Scheduler | Behavior | Usage |
| :--- | :--- | :--- |
| `StepLR` | Decays LR every N steps | Stable, predictable decay |
| `ReduceLROnPlateau` | Decays when a metric stops improving | Adaptive training stop/slow |
| `CosineAnnealingLR` | Cycles LR in a cosine curve | SOTA for vision modeling |

In [ ]:
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=30, gamma=0.1)

# Call in training loop after optimizer.step()
# scheduler.step()

### Common Pitfalls
- **Logits vs Probabilities**: Many PyTorch losses (CrossEntropy, BCEWithLogits) expect unnormalized logits. Do not apply Softmax/Sigmoid manually before the loss.
- **Zero Grad**: If you forget `optimizer.zero_grad()`, gradients will accumulate from the previous step, leading to training divergence.
- **Dtypes for Loss**: `CrossEntropyLoss` expects `torch.long` for discrete class indices; `BCEWithLogitsLoss` expects `torch.float`.

### Key Takeaways
- Always prefer `AdamW` for its robust handling of sparse gradients.
- Logits are raw scores from the last linear layer; use `nn.BCEWithLogitsLoss` or `nn.CrossEntropyLoss` to handle numerical stability internally.
- Learning rate is the most critical hyperparameter to tune.